<a href="https://colab.research.google.com/github/jbeeksieu2023-del/ML-fundamentals-2026/blob/main/assignment_1_jade_beeksieu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GitHub repository: https://github.com/jbeeksieu2023-del/ML-fundamentals-2026

**Task: Identifying the Prediction Target
Lecture material: Lecture 1 (Problem Formulation), Lecture 2 (Data Inspection).
• Inspect the dataset and identify which column should be treated as the target variable for this assignment.
• Justify why this column represents the appropriate prediction objective in the context of the marketing
campaign.
• Identify at least two other variables that could superficially appear to be valid targets and explain why they
should not be treated as the prediction objective.**

It is important to first diagnose the business problem, to define a goal for our model. This means that we find the **right** solution to the goal.

**"Given client and campaign information available at the time of contact, predict whether
the client subscribes to a term deposit (i.e., a type of short-term investment)."**


From our business problem formulated above, our target variable wil be the column names "y".
This column answers booleanly, yes/no, to whether or not the clients have subscribed to a term deposit. This is a clear answer to the business question, and will allow us to predict based on marketing campaigns, its success directly - yes they have commited, no they have not.
Another variable which may appear as a target would be ...................


In [17]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

# Models and baseline
!pip install scikit-optimize

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay,
)

# Bayesian Optimization
from skopt import BayesSearchCV
sns.set_theme(style="whitegrid")

# Suppress benign sklearn internal warnings
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")

In [18]:
def compute_metrics(y_true, y_pred, y_score=None):
    """Return a dict of standard classification metrics.
    y_score: predicted probabilities for the positive class (needed for ROC-AUC).
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_score)
    return metrics


def display_dataset_split(X_train, y_train, X_val, y_val, X_test, y_test):
    df = pd.DataFrame({
        "Set": ["Train", "Validation", "Test"],
        "X shape": [X_train.shape, X_val.shape, X_test.shape],
        "y shape": [y_train.shape, y_val.shape, y_test.shape],
    })
    display(df.style.hide(axis="index"))


def display_metrics(name, metrics, decimals=4):
    print(f"\n{name}")
    df = pd.DataFrame([metrics]).round(decimals)
    display(df.style.hide(axis="index"))


def display_model_results(name, best_params, metrics, decimals=4):
    print(f"\n{name} — Best Parameters")
    display(pd.DataFrame([best_params]).style.hide(axis="index"))
    print(f"{name} — Validation Metrics")
    display(pd.DataFrame([metrics]).round(decimals).style.hide(axis="index"))


# Accumulator for the summary table
results_table = []

def add_result(name, split, metrics, best_params=None, notes=None):
    row = {
        "Model": name,
        "Split": split,
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1": metrics.get("f1"),
        "ROC-AUC": metrics.get("roc_auc"),
        "Notes": notes,
    }
    if best_params:
        row["Best Params"] = ", ".join(f"{k}={v}" for k, v in best_params.items())
    else:
        row["Best Params"] = None
    results_table.append(row)


def show_results(sort_by=("Split", "ROC-AUC"), ascending=(True, False), decimals=4):
    df = pd.DataFrame(results_table)
    if df.empty:
        print("results_table is empty.")
        return df
    df_sorted = df.sort_values(list(sort_by), ascending=list(ascending))
    display(df_sorted.round(decimals).style.hide(axis="index"))
    return df_sorted

**Task: Data Loading and Exploration
Lecture material: Lecture 1 (Problem Formulation), Lecture 2 (Data Inspection and EDA).
• Load the dataset into a Pandas DataFrame.
• Inspect the structure of the dataset: number of observations, number of features, data types, and basic
summary statistics.
• Identify which variables are numerical and which are categorical.
Individual Assignment I AI: Machine Learning Foundation
• Analyze the distribution of the target variable and comment on potential class imbalance.
• Detect explicit and implicit missing values (e.g., special categories such as unknown).
• Visualize the distribution of at least:
 –  two numerical variables; and
 –  two categorical variables.
• Identify at least one variable that may require special consideration before modeling (e.g., due to distributional
properties, extreme skewness, or availability at prediction time), and briefly justify your reasoning.
Note: Exploratory analysis is not a checklist of plots. Each visualization or statistic should support a specific
observation or hypothesis about the data.**

In [19]:
import pandas

In [29]:
# First, download the dataset from the GitHub repository
!wget -q https://raw.githubusercontent.com/jbeeksieu2023-del/ML-fundamentals-2026/main/data/bank-additional.csv

# Now, read the downloaded CSV file
df=pandas.read_csv("bank-additional.csv", sep=";")

FileNotFoundError: [Errno 2] No such file or directory: 'bank-additional.csv'

In [ ]:
df.head()

In [ ]:
print(f"Number of observations (rows): {df.shape[0]}")
print(f"Number of features (columns): {df.shape[1]}")

In [ ]:
print("Data types and non-null counts:")
df.info()

In [ ]:
print("Basic summary statistics for numerical columns:")
display(df.describe())

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

We can gather some basic information about the dataset now. We have 4119 rows and 21 columns. From the columns, we have split them into two data types, numerical and categorial, shown above.

In [ ]:
# Distribution of the target variable 'y'
print("Distribution of target variable 'y':")
display(df['y'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x='y', data=df, hue='y', legend=False)
plt.title('Distribution of Target Variable (y)')
plt.xlabel('Subscribed to Term Deposit')
plt.ylabel('Count')
plt.show()

# Check for class imbalance
class_imbalance = df['y'].value_counts(normalize=True) * 100
print("\nPercentage distribution of 'y':")
display(class_imbalance)

In [ ]:
# Detect explicit missing values (NaN)
print("Explicit missing values (NaN) per column:")
display(df.isnull().sum())

# Detect implicit missing values (e.g., 'unknown') in categorical columns
print("\nImplicit missing values ('unknown') in categorical columns:")
for col in df.select_dtypes(include='object').columns:
    if 'unknown' in df[col].unique():
        print(f"  - Column '{col}': {df[col].value_counts().get('unknown', 0)} occurrences")

In [ ]:
# Visualize distributions of two numerical variables
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['age'], kde=True)
plt.title('Distribution of age')

plt.subplot(1, 2, 2)
sns.histplot(df['duration'], kde=True)
plt.title('Distribution of Duration')

plt.tight_layout()
plt.show()

# Visualize distributions of two categorical variables
plt.figure(figsize=(16, 5))

plt.subplot(1, 2, 1)
sns.countplot(y='job', data=df, order=df['job'].value_counts().index, palette='magma', hue='job', legend=False)
plt.title('Distribution of Job')

plt.subplot(1, 2, 2)
sns.countplot(x='poutcome', data=df, palette='cividis', hue='poutcome', legend=False)
plt.title('Distribution of Outcome')

plt.tight_layout()
plt.show()

Given the visualisations above, duration appears to require special consideration due to its extreme right-skewness. It is also important to consider potential data leakage from the duration variable. Call duration would have been obtained after the call, meaning potentially  already knowing the clients' decision. This could be labelled as temporal leakage since the information of call duration is not available at prediction time.

The default variable has the highest unknown entries. It must be considered as potential implicit missingness, and therefore imputed or dealt with accordingly.

Also, as seen from the csv file, pdays has multiple 999 entries. The value seems to suggest it is a code instead of a numeric value, and if this is not considered - it could be very misleading. Visualising the data may help to consider best-next options, as done below.

In [ ]:
plt.figure()
plt.hist(df["pdays"], bins=50)
plt.title("Distribution of pdays (all values)")
plt.xlabel("pdays")
plt.ylabel("count")
plt.show()

As expected, there is a large amount of 999 entries. The distribution is heavily dominated. A visualisation of the data excluding the entry 999 may provide a more meaningful distribution, as created below.

In [ ]:
non_999 = df.loc[df["pdays"] != 999, "pdays"]

plt.figure()
plt.hist(non_999, bins=30)
plt.title("Distribution of pdays (excluding 999)")
plt.xlabel("pdays (non-999)")
plt.ylabel("count")
plt.show()

Results may suggest that 999 is a sentinel code, rather than a number of days since previously contacted. If we treat is as a continous variable, huge outliers will appear and the model will be biased. This variable needs to be hevaily considered, and potentially manipulated.

**Task: Task Ordering
Lecture material: Lecture 2 (Data Splitting and Leakage), Lecture 5 (Preprocessing), Lecture 9 (ML Pipeline).
• Determine the correct order in which the data preparation tasks in this assignment should be performed.
• Provide a structured justification for your chosen order.
• For each step in your proposed sequence, explain:
– what information is allowed to be used at that stage;
– what information must not be used;
– what type of data leakage could occur if the order were changed.
• Discuss at least one example of an incorrect ordering and explain the consequences it would have on model
evaluation.**

**Task: Data Splitting Lecture material: Lecture 2 (Data Splitting and Leakage), Lecture 9 (ML Pipeline).
• Split the dataset into training, validation, and test sets.
• Justify your choice of proportions for each split.
• Perform stratified splitting with respect to the target variable and explain why stratification is necessary for
this dataset.
• Clearly describe at which stage of your pipeline the split must occur, and explain what types of data leakage
would arise if splitting were performed later.
Note: A recommended strategy is to first split the dataset into a training set and a temporary set, and then
split the temporary set into validation and test sets. Use the stratify argument of train test split where
appropriate.**

In [ ]:
# Separate features (X) and target (y)
X = df.drop('y', axis=1)
y = df['y']

# First split: Create training set and a temporary set (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: Divide the temporary set into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Display the shapes of the splits
print("Dataset splits:")
display_dataset_split(X_train, y_train, X_val, y_val, X_test, y_test)

# Verify stratification for the target variable in each split
print("\nTarget variable distribution in each split:")
print("Train set 'y' distribution:")
display(y_train.value_counts(normalize=True))
print("\nValidation set 'y' distribution:")
display(y_val.value_counts(normalize=True))
print("\nTest set 'y' distribution:")
display(y_test.value_counts(normalize=True))

Since the dataset is of medium-size, the choice of split would usually range between a 80/20/20 or a 70/15/15. Using the rule of thumb that there should be 12 per class minimum, and 100 to be reliable for the validation and test sets. Since the 'y' target only has around 10% positive values, splitting them using 80/20/20 would leave the validation/test sets with only 45 positive values. Hence, the 70/15/15 splitting proportions were chosen so that our future statistics are reliable.
Since our target variable 'y' is imbalanced, approx 10% are positive, we must use stratification. Each split will have the same proportion of positive and negative values for the 'y' target variable. Our evaluation of the model therefore will mantain reliable and unbiased, which it would otherwise compromise.

**Task: Managing Missing Values
Lecture material: Lecture 2 (Data Inspection), Lecture 5 (Preprocessing and Pipeline Discipline).
• Identify both explicit missing values (e.g., NaN) and implicit missing values (e.g., categories such as unknown
or sentinel numerical values, i.e., values that may represent special codes rather than genuine measurements).
• Quantify the extent of missingness for each affected variable.
• Propose and justify a strategy for handling missing values in each case (e.g., removal, imputation, separate
category, indicator variable).
• Clearly state which operations must be fitted using the training set only, and explain why.
Note: Your strategy should distinguish between “data cleaning” decisions (e.g., correcting inconsistent entries)
and “modeling” decisions (e.g., whether missingness itself may carry predictive information).**

I have already gathered explicit and implicit missing values.
There has shown to be no explicit missing values in any columns. However, implicit missing values are present in the following extent:

---
"Implicit missing values ('unknown') in categorical columns:
  - Column 'job': 39 occurrences
  - Column 'marital': 11 occurrences
  - Column 'education': 167 occurrences
  - Column 'default': 803 occurrences
  - Column 'housing': 105 occurrences
  - Column 'loan': 105 occurrences"

---




job: 39 (≈ 0.95%)

marital: 11 (≈ 0.27%)

education: 167 (≈ 4.05%)

default: 803 (≈ 19.5%)

housing: 105 (≈ 2.55%)

loan: 105 (≈ 2.55%)

Job and Marital

---


Both of these categorical variables show a low % of missingness. Since dropping these rows may risk deleting the few positive target 'y' variables that we have, we can keep them as a separate category.

Loan and Housing


---

Although these categorical variables have a moderately higher % of implicit missingness, imputing these value as "no" is assuming client financial attributes to satisfy our model and will have an impact on the predictive power and reliability of our model. These "unknown"'s remain as a separate category, too.

Education

---

This variable , for the same reasons, will keep "unknown" as a separate category.

Default

---

The "default" variable has the same fate. A missing rate so large cannot be caused by randomness, so we cannot just drop them as they may be informative in themselves. A popular imputing option would be model imputation, replacing it with the most common value, but this will have large effects on the model and remove potentially informative missingness. For this reason, we create an explicit category.

Renaming the implicit missingness of the variables above to "missing" is the data cleaning part. We ensure that missing values are represented the same way, to be  consistent across all variables. The decision to not impute, but treat this implicit missingness as separate category is a modelling decision, so that our model does not become bias due to potential removals of low positive 'y''s.

In [ ]:
X_train_clean = X_train.copy()
X_val_clean   = X_val.copy()
X_test_clean  = X_test.copy()


In [ ]:
unknown_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']
print(f"unknown_cols defined: {unknown_cols}")

In [ ]:

for c in unknown_cols:
    for split in (X_train_clean, X_val_clean, X_test_clean):
        split[c] = split[c].replace("unknown", "missing")

Pdays

---

As mentioned before, this variable required special considerations so we review it now. Even though it does not show up in the variables with implicit missing values, we must treat 999 as a code. Due to context, it can be interpreted as the clients having no previous contact. The next step is to model this sentinel meaning. First, we should clean the column so that the entry "999" becomes a missing value, N/A. However, we don't want to lose the data by doing this. A new boolean variable should be created, recording whether or not the client has been previously contacted or not.

In [ ]:
def add_pdays_features(X):
    X = X.copy()
    X["previously_contacted"] = (X["pdays"] != 999).astype(int)
    X["pdays_clean"] = X["pdays"].replace(999, pd.NA)
    return X

X_train_clean = add_pdays_features(X_train_clean)
X_val_clean   = add_pdays_features(X_val_clean)
X_test_clean  = add_pdays_features(X_test_clean)

**Task: Encoding Categorical Variables
Lecture material: and Expressiveness).
Lecture 4 (Categorical Encoding), Lecture 6 (Linear Models), Lecture 9 (Feature Engineering
• Identify all categorical variables in the dataset.
• Distinguish between nominal variables (categories without intrinsic order, e.g., job type) and ordinal variables
(categories with a meaningful order, e.g., education level), and justify your classification.
• Select and apply an appropriate encoding strategy for each categorical variable.
• Clearly state which encoders must be fitted on the training set only, and explain why.
• Analyze how encoding changes:
– the dimensionality of the dataset;
– the interpretability of model coefficients;
– the types of decision boundaries a linear model can represent.
Note: Encoding is not a purely mechanical transformation. Your justification should explicitly connect your encoding
decisions to the assumptions and behavior of Logistic Regression.**

In [ ]:
print("Categorical columns:", cat_cols)

Nominal variables:

---



Job
Marital
Default
Housing
Loan
Contact
Poutcome

These are all nominal variables. There is no meaningful order in their respectful categories.


Ordinal variables:

---
Education:
The order of its' categories are based on level of education. They are ordered as such:
university.degree > professional.course > high.school > basic.9y > basic.6y > basic.4y. The category called "missing" has no place in this order.

Month and day_of_week: There is chronological order here but they are not ordinal variables. If we encode them as such, the model will interpret than December (month), or Sunday (week), are far away from January (month), or Monday(week). They go in a cyclic order.




Encoding

---

Job
Marital
Default
Housing
Loan
Contact
Poutcome
. All of these nominal variables will be encoded using one-hot-encoding since they have no meaningful order to represent. One-hot coding does not create any ranks or order, hence its suitability here.


---


Education
. After consideration, although ordinal, encoding the education variable as such creates risk. It would assume that there i the same jump between each education level, which is a bold assumption. We also do not know the effect higher education will have on subscribing. Importantly, the new category "missing" does not have a position on the order of education so cannot be treated as such. One-hot encoding here will separate each education level, and the model will still be able to learn the effects that each level, independently, has on subscribing.

---


Month
Day_of_week
. Although in chronological order, its cyclical nature does not allow us to encode them with ranking or the model will misunderstand. However, sin/cos encoding could be applied. This will mantain circular distance and convert into two synthetic features.



In [ ]:


# 0-based indices are easiest for cyclical encoding
MONTH_TO_IDX = {
    "jan": 0, "feb": 1, "mar": 2, "apr": 3, "may": 4, "jun": 5,
    "jul": 6, "aug": 7, "sep": 8, "oct": 9, "nov": 10, "dec": 11
}

DOW_TO_IDX = {
    "mon": 0, "tue": 1, "wed": 2, "thu": 3, "fri": 4
    # dataset day_of_week is usually Mon–Fri only
}

def add_cyclical_time_features(X):
    X = X.copy()

    # Map strings -> indices
    m = X["month"].str.lower().map(MONTH_TO_IDX)
    d = X["day_of_week"].str.lower().map(DOW_TO_IDX)

    # Optional: handle unexpected/missing values (shouldn't happen, but humans happen)
    X["month_missing"] = m.isna().astype(int)
    X["dow_missing"] = d.isna().astype(int)
    m = m.fillna(0)
    d = d.fillna(0)

    # Encode with sine/cosine
    month_period = 12
    dow_period = 5

    X["month_sin"] = np.sin(2 * np.pi * m / month_period)
    X["month_cos"] = np.cos(2 * np.pi * m / month_period)
    X["dow_sin"]   = np.sin(2 * np.pi * d / dow_period)
    X["dow_cos"]   = np.cos(2 * np.pi * d / dow_period)

    # Drop original categorical columns (since we've encoded them)
    X = X.drop(columns=["month", "day_of_week"])

    return X

In [ ]:
X_train_time = add_cyclical_time_features(X_train_clean)
X_val_time   = add_cyclical_time_features(X_val_clean)
X_test_time  = add_cyclical_time_features(X_test_clean)

X_train_time[["month_sin","month_cos","dow_sin","dow_cos"]].head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(X_train_time["month_cos"], X_train_time["month_sin"], alpha=0.3)
plt.title("Cyclical encoding: month (cos vs sin)")
plt.xlabel("month_cos")
plt.ylabel("month_sin")
plt.show()